In [1]:
print("ok")

ok


In [3]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

def list_my_models():
    # 1. Load your key
    load_dotenv()
    api_key = os.getenv("GEMINI_API_KEY")
    
    if not api_key:
        print("Error: No API key found in environment variables.")
        return

    # 2. Configure the SDK
    genai.configure(api_key=api_key)

    print("--- Available Gemini Models for your Key ---")
    try:
        # 3. List all models and filter for 'generateContent' capability
        for m in genai.list_models():
            if 'generateContent' in m.supported_generation_methods:
                print(f"Model Name: {m.name}")
                print(f"  > Display Name: {m.display_name}")
                print(f"  > Description: {m.description}\n")
    except Exception as e:
        print(f"Failed to list models: {e}")

# Run the function
list_my_models()

--- Available Gemini Models for your Key ---


Model Name: models/gemini-2.5-flash
  > Display Name: Gemini 2.5 Flash
  > Description: Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.

Model Name: models/gemini-2.5-pro
  > Display Name: Gemini 2.5 Pro
  > Description: Stable release (June 17th, 2025) of Gemini 2.5 Pro

Model Name: models/gemini-2.0-flash
  > Display Name: Gemini 2.0 Flash
  > Description: Gemini 2.0 Flash

Model Name: models/gemini-2.0-flash-001
  > Display Name: Gemini 2.0 Flash 001
  > Description: Stable version of Gemini 2.0 Flash, our fast and versatile multimodal model for scaling across diverse tasks, released in January of 2025.

Model Name: models/gemini-2.0-flash-lite-001
  > Display Name: Gemini 2.0 Flash-Lite 001
  > Description: Stable version of Gemini 2.0 Flash-Lite

Model Name: models/gemini-2.0-flash-lite
  > Display Name: Gemini 2.0 Flash-Lite
  > Description: Gemini 2.0 Flash-Lite

Model Name: models/gemini-2.5-flash

In [4]:
import os
from dotenv import load_dotenv
import langchain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

load_dotenv()
GEMINI_API_KEY=os.getenv("GEMINI_API_KEY")

def initiatellm():
    try:
     llm= ChatGoogleGenerativeAI(
    model="models/gemini-2.0-flash",
    temperature=0.7,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key=GEMINI_API_KEY
)
     return llm
    except Exception as e:
     print(f"Error message:{e.message}")
     return None

In [5]:
llm=initiatellm()
response = llm.invoke("What is the capital of France?")
print(response.content)

ChatGoogleGenerativeAIError: Error calling model 'models/gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 52.193569498s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas

In [14]:
import pandas as pd
from pathlib import Path

def profile_data(file_path: str) -> dict:
 try:
    ext= Path(file_path).suffix
    
    if ext == '.csv':
            df = pd.read_csv(file_path, encoding='utf-8')
    elif ext == '.xlsx':
            df = pd.read_excel(file_path)
    else:
            ValueError("Unsupported file format. Please upload a CSV or Excel file.")
            return None, None, None

    profile = {}

    # Basic info
    profile["shape"] = df.shape
    profile["columns"] = list(df.columns)
    profile["dtypes"] = df.dtypes.astype(str).to_dict()

    # Missing values
    profile["missing_values"] = df.isnull().sum().to_dict()

    # Unique values
    profile["unique_values"] = df.nunique().to_dict()
    # info
    # profile["info"] = df.info()

    # Summary statistics
    profile["summary_stats"] = df.describe(include="all").fillna("").to_dict()

    # Correlation (only numeric)
    numeric_df = df.select_dtypes(include="number")
    if not numeric_df.empty:
        profile["correlation"] = numeric_df.corr().to_dict()
    else:
        profile["correlation"] = {}

    return profile
 except Exception as e:
    print(f"Error profiling data: {e}")
    return None

def profile_to_text(profile: dict) -> str:
    """Convert profile dict into text for LLM"""
    text = []

    text.append(f"Dataset shape: {profile['shape']}")
    text.append(f"Columns: {profile['columns']}")
    text.append(f"Data types: {profile['dtypes']}")

    text.append("Missing values:")
    text.append(str(profile["missing_values"]))

    text.append("Unique values:")
    text.append(str(profile["unique_values"]))

    text.append("Summary stats:")
    text.append(str(profile["summary_stats"]))

    text.append("Correlations:")
    text.append(str(profile["correlation"]))

    return "\n".join(text)

In [15]:
def run_profiling_pipeline(file_path: str):
    # Step 1: Profile dataset
    profile = profile_data(file_path)

    # Step 2: Convert to text for LLM
    summary_text = profile_to_text(profile)
    

    # Step 3: Generate AI insights
    insights = generate_insights(summary_text)

    return {
        "profile": profile,
        "insights": insights
    }
print(run_profiling_pipeline("./insurance.csv"))
# print(type(run_profiling_pipeline("./insurance.csv")))

APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits', 'code': 402}}

In [ ]:
def test_llm(input:str)->str:
       prompt = PromptTemplate(
       input_variables=["input"],
       template="""
       you are an intelligent assistant ,your goal is to answer to user question in short answer not more 
       than 200 characters
       {input}
       """)
       chain=prompt|llm
       response=chain.invoke({"input":input})
       return response.content
print(test_llm("What is the capital of France?"))